# Tax Planning Exo-Brain - Baby Step 0
## Build the dual-universe synthetic vault

This notebook adapts the Civil Litigation Exo-Brain benchmark to multi-jurisdictional tax planning. It creates 100 synthetic tax-code modules and 10 complex synthetic conglomerates. **No recommendation is produced at Step 0.**

All jurisdictions, rules, rates, entities, and transactions are fictional. This is not tax or legal advice.

## Design principle

The external rule universe and internal company universe are stored separately, linked by stable identifiers, and governed by DEC-000. Evidence may later narrow reliance; only a human decision may widen authority.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
OUTPUT_ROOT = Path('/content/drive/MyDrive/Tax_Planning_ExoBrain_Project/Step_0')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
VAULT_PATH = OUTPUT_ROOT / 'Tax_Planning_ExoBrain_Vault'
print(f'Output folder: {OUTPUT_ROOT}')


In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
import random
import shutil
import textwrap
from datetime import date
from pathlib import Path


SEED = 20260721
TODAY = date(2026, 7, 21).isoformat()
PROJECT_NAME = "Tax_Planning_ExoBrain"


JURISDICTIONS = [
    ("JUR-001", "Aurelia", "Continent Alpha", "territorial", "AUR"),
    ("JUR-002", "Borealis", "Continent Alpha", "worldwide", "BOR"),
    ("JUR-003", "Caledon", "Continent Beta", "participation exemption", "CAL"),
    ("JUR-004", "Demeria", "Continent Beta", "territorial", "DEM"),
    ("JUR-005", "Elandra", "Continent Gamma", "worldwide", "ELA"),
    ("JUR-006", "Faron", "Continent Gamma", "remittance hybrid", "FAR"),
    ("JUR-007", "Galenia", "Continent Delta", "participation exemption", "GAL"),
    ("JUR-008", "Helios", "Continent Delta", "territorial", "HEL"),
    ("JUR-009", "Ilyria", "Continent Epsilon", "worldwide", "ILY"),
    ("JUR-010", "Juno", "Continent Epsilon", "participation exemption", "JUN"),
    ("JUR-011", "Kestrel", "Continent Zeta", "territorial", "KES"),
    ("JUR-012", "Lydon", "Continent Zeta", "worldwide", "LYD"),
    ("JUR-013", "Meridia", "Continent Eta", "participation exemption", "MER"),
    ("JUR-014", "Nereus", "Continent Eta", "territorial", "NER"),
    ("JUR-015", "Orinth", "Continent Theta", "worldwide", "ORI"),
    ("JUR-016", "Palladia", "Continent Theta", "participation exemption", "PAL"),
    ("JUR-017", "Quillon", "Continent Iota", "territorial", "QUI"),
    ("JUR-018", "Rhea", "Continent Iota", "worldwide", "RHE"),
    ("JUR-019", "Solara", "Continent Kappa", "participation exemption", "SOL"),
    ("JUR-020", "Tirsen", "Continent Kappa", "territorial", "TIR"),
]

CODE_MODULES = [
    ("CIT", "Corporate income tax and loss utilization"),
    ("WHT", "Withholding taxes, treaty relief, and repatriation"),
    ("TP", "Transfer pricing and related-party transactions"),
    ("CFC", "Controlled foreign companies and anti-avoidance"),
    ("GMT", "Minimum tax, incentives, substance, and reporting"),
]

GROUPS = [
    ("CONG-001", "Asterion Advanced Manufacturing", "Advanced manufacturing"),
    ("CONG-002", "BlueRiver Digital Services", "Digital platforms"),
    ("CONG-003", "Cobalt Life Sciences", "Life sciences"),
    ("CONG-004", "Delta Consumer Holdings", "Consumer products"),
    ("CONG-005", "Evergreen Infrastructure", "Infrastructure"),
    ("CONG-006", "Frontier Energy Systems", "Renewable energy"),
    ("CONG-007", "Granite Financial Technologies", "Financial technology"),
    ("CONG-008", "Horizon Logistics Network", "Logistics"),
    ("CONG-009", "Ionis Media and Data", "Media and data"),
    ("CONG-010", "Juniper Industrial Software", "Enterprise software"),
]

ENTITY_ROLES = [
    "Ultimate parent", "Regional holding", "Operating company", "IP owner",
    "R&D center", "Distribution principal", "Limited-risk distributor",
    "Shared-services center", "Treasury company", "Financing vehicle",
    "Procurement hub", "Manufacturing principal", "Contract manufacturer",
]

TRANSACTION_TYPES = [
    "royalty", "management services", "intercompany financing", "goods supply",
    "software license", "cost contribution", "guarantee fee", "dividend",
]


def slug(value: str) -> str:
    return "".join(c.lower() if c.isalnum() else "-" for c in value).strip("-").replace("--", "-")


def write_text(path: Path, content: str) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content.rstrip() + "\n", encoding="utf-8")


def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        raise ValueError(f"No rows for {path}")
    with path.open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def build_vault(vault: Path) -> dict:
    rng = random.Random(SEED)
    if vault.exists():
        shutil.rmtree(vault)
    directories = [
        "00_System", "01_Jurisdictions", "02_Tax_Codes", "03_Conglomerates",
        "04_Entities", "05_Transactions", "06_Sources", "07_Atomic_Claims",
        "08_Contradictions", "09_Recommendations", "10_Decisions",
        "11_Quarterly_Updates", "12_Reports", "13_Audit", "14_Hot_Cache",
        "15_Application", "16_Data", "17_Notebooks",
    ]
    for folder in directories:
        (vault / folder).mkdir(parents=True, exist_ok=True)

    jurisdiction_rows = []
    tax_code_rows = []
    for j_index, (jid, name, region, system, currency) in enumerate(JURISDICTIONS, 1):
        treaty_partners = [j[1] for j in JURISDICTIONS if j[0] != jid]
        rng.shuffle(treaty_partners)
        partners = sorted(treaty_partners[: rng.randint(5, 10)])
        headline_rate = rng.choice([12.5, 15.0, 18.0, 20.0, 22.0, 24.0, 25.0, 27.5, 30.0])
        jurisdiction_rows.append({
            "jurisdiction_id": jid, "name": name, "region": region,
            "tax_system": system, "currency": currency,
            "headline_cit_rate_pct": headline_rate,
            "treaty_partner_count": len(partners),
            "synthetic": True, "effective_from": "2026-Q3",
        })
        write_text(vault / "01_Jurisdictions" / f"{jid}_{slug(name)}.md", f"""
---
object_type: jurisdiction
jurisdiction_id: {jid}
name: {name}
region: {region}
tax_system: {system}
currency: {currency}
headline_cit_rate_pct: {headline_rate}
effective_from: 2026-Q3
synthetic: true
---

# {name}

Synthetic jurisdiction profile for architecture testing only.

## Treaty network

{', '.join(partners)}.

## Linked tax-code modules

""" + "\n".join(f"- [[TC-{j_index:02d}-{m_index:02d}]]" for m_index in range(1, 6)))

        for m_index, (module, title) in enumerate(CODE_MODULES, 1):
            cid = f"TC-{j_index:02d}-{m_index:02d}"
            rate = round(max(0.0, min(35.0, headline_rate + rng.uniform(-8, 8))), 2)
            threshold = rng.choice([0.5, 1, 5, 10, 20, 50, 100, 750])
            change_sensitivity = rng.choice(["low", "medium", "high"])
            rule = {
                "CIT": f"Taxable operating profit is subject to a synthetic {headline_rate}% headline rate, with modeled loss-use and interest-limitation constraints.",
                "WHT": f"Outbound related-party payments are subject to a synthetic base rate of {rate}%, potentially reduced by treaty, beneficial-ownership, and substance tests.",
                "TP": f"Controlled transactions above {threshold} million {currency} require contemporaneous arm's-length support and a synthetic local-file analysis.",
                "CFC": f"Low-taxed controlled income may be included at parent level where the modeled effective rate is below {rate}% and no substance exception applies.",
                "GMT": f"Groups above the synthetic {threshold} million consolidated-revenue threshold enter the minimum-tax, incentive, substance, and reporting module.",
            }[module]
            tax_code_rows.append({
                "tax_code_id": cid, "jurisdiction_id": jid, "jurisdiction": name,
                "module": module, "title": title, "version": "2026-Q3-V001",
                "effective_from": "2026-Q3", "numeric_parameter": rate,
                "threshold_million_local": threshold,
                "change_sensitivity": change_sensitivity,
                "status": "CURRENT", "synthetic": True,
            })
            write_text(vault / "02_Tax_Codes" / f"{cid}_{module}.md", f"""
---
object_type: tax_code
tax_code_id: {cid}
jurisdiction: "[[{jid}_{slug(name)}]]"
module: {module}
version: 2026-Q3-V001
effective_from: 2026-Q3
status: CURRENT
change_sensitivity: {change_sensitivity}
synthetic: true
---

# {cid} - {title}

## Synthetic rule

{rule}

## Decision variables

- Numeric parameter: {rate}
- Threshold: {threshold} million {currency}
- Substance and documentation remain mandatory constraints.

## Governance

This record is fictional, is not a statement of any real jurisdiction's law, and may not be used as tax advice.
""")

    group_rows, entity_rows, ownership_rows, transaction_rows = [], [], [], []
    for g_index, (gid, group_name, industry) in enumerate(GROUPS, 1):
        selected_js = rng.sample(JURISDICTIONS, rng.randint(6, 9))
        entity_count = rng.randint(13, 18)
        group_rows.append({
            "conglomerate_id": gid, "name": group_name, "industry": industry,
            "home_jurisdiction_id": selected_js[0][0],
            "jurisdiction_count": len(selected_js), "entity_count": entity_count,
            "consolidated_revenue_usd_m": rng.randint(900, 42000),
            "recommendation_status": "NOT_DEVELOPED",
            "recommendation_version": "NONE", "synthetic": True,
        })
        group_entities = []
        for e_index in range(1, entity_count + 1):
            eid = f"ENT-{g_index:02d}-{e_index:03d}"
            j = selected_js[(e_index - 1) % len(selected_js)]
            role = "Ultimate parent" if e_index == 1 else rng.choice(ENTITY_ROLES[1:])
            revenue = rng.randint(5, 3000)
            employees = rng.randint(2, 4200)
            substance = "high" if employees > 400 else ("medium" if employees > 50 else "low")
            row = {
                "entity_id": eid, "conglomerate_id": gid,
                "legal_name": f"{group_name.split()[0]} {role} {e_index}",
                "jurisdiction_id": j[0], "jurisdiction": j[1], "role": role,
                "revenue_usd_m": revenue, "employees": employees,
                "substance_indicator": substance,
                "intangibles_owner": role == "IP owner",
                "financing_function": role in {"Treasury company", "Financing vehicle"},
                "synthetic": True,
            }
            entity_rows.append(row)
            group_entities.append(row)
            if e_index > 1:
                parent_index = rng.randint(1, max(1, e_index - 1))
                ownership_rows.append({
                    "edge_id": f"OWN-{g_index:02d}-{e_index:03d}",
                    "parent_entity_id": f"ENT-{g_index:02d}-{parent_index:03d}",
                    "child_entity_id": eid,
                    "ownership_pct": rng.choice([51, 60, 75, 80, 90, 95, 100]),
                    "effective_from": "2026-Q3", "synthetic": True,
                })

        for t_index in range(1, rng.randint(18, 28) + 1):
            payer, recipient = rng.sample(group_entities, 2)
            transaction_rows.append({
                "transaction_id": f"TX-{g_index:02d}-{t_index:03d}",
                "conglomerate_id": gid,
                "payer_entity_id": payer["entity_id"],
                "recipient_entity_id": recipient["entity_id"],
                "transaction_type": rng.choice(TRANSACTION_TYPES),
                "annual_amount_usd_m": round(rng.uniform(1, 180), 2),
                "currency": "USD", "related_party": True,
                "support_status": rng.choice(["documented", "partial", "to_be_tested"]),
                "synthetic": True,
            })

        entity_table = "\n".join(
            f"| [[{e['entity_id']}]] | {e['role']} | [[{e['jurisdiction_id']}_{slug(e['jurisdiction'])}]] | {e['substance_indicator']} |"
            for e in group_entities
        )
        linked_codes = sorted({f"TC-{int(e['jurisdiction_id'].split('-')[1]):02d}-{m:02d}" for e in group_entities for m in range(1, 6)})
        write_text(vault / "03_Conglomerates" / f"{gid}_{slug(group_name)}.md", f"""
---
object_type: conglomerate
conglomerate_id: {gid}
name: {group_name}
industry: {industry}
baseline_quarter: 2026-Q3
recommendation_status: NOT_DEVELOPED
recommendation_version: NONE
synthetic: true
---

# {group_name}

## Step 0 status

The group is represented but no tax structure has been recommended. Observation, organization, and validation are the only permitted operations.

## Entity inventory

| Entity | Role | Jurisdiction | Substance |
|---|---|---|---|
{entity_table}

## Relevant tax-code universe

""" + "\n".join(f"- [[{code}]]" for code in linked_codes) + """

## Open questions for later steps

- Which flows create material tax leakage or double taxation?
- Which structures satisfy substance, transfer-pricing, anti-avoidance, and minimum-tax constraints?
- Which alternative is most robust to quarterly code changes?
- What human approvals are required before any recommendation is accepted?
""")

    for entity in entity_rows:
        write_text(vault / "04_Entities" / f"{entity['entity_id']}.md", f"""
---
object_type: entity
entity_id: {entity['entity_id']}
conglomerate_id: {entity['conglomerate_id']}
jurisdiction_id: {entity['jurisdiction_id']}
role: {entity['role']}
substance_indicator: {entity['substance_indicator']}
synthetic: true
---

# {entity['legal_name']}

- Group: [[{entity['conglomerate_id']}]]
- Jurisdiction: [[{entity['jurisdiction_id']}_{slug(entity['jurisdiction'])}]]
- Employees: {entity['employees']}
- Revenue: USD {entity['revenue_usd_m']} million
- Intangibles owner: {entity['intangibles_owner']}
- Financing function: {entity['financing_function']}
""")

    write_csv(vault / "16_Data" / "jurisdictions.csv", jurisdiction_rows)
    write_csv(vault / "16_Data" / "tax_codes.csv", tax_code_rows)
    write_csv(vault / "16_Data" / "conglomerates.csv", group_rows)
    write_csv(vault / "16_Data" / "entities.csv", entity_rows)
    write_csv(vault / "16_Data" / "ownership_edges.csv", ownership_rows)
    write_csv(vault / "16_Data" / "intercompany_transactions.csv", transaction_rows)

    roadmap = """# Tax Planning Exo-Brain - Baby Steps 0-10

The sequence adapts the civil-litigation benchmark to a synthetic, multi-jurisdictional tax-planning operating system.

| Step | Capability | Principal output | Governance boundary |
|---:|---|---|---|
| 0 | Build the dual-universe vault | 100 tax-code modules, 10 conglomerates, entity and transaction graphs | Observe and organize only; no recommendation |
| 1 | Execute the first structure-design loop | Recommendation V1 for each conglomerate, supporting and contrary rule sets | Internal analytical baseline only |
| 2 | Test generalization across business models | Conglomerate-specific objectives, constraints, sensitivity, fragility | Qualify or reopen internally |
| 3 | Add provenance, atomic tax claims, and contradiction control | Source records, claims, confidence, dependencies, contradictions | Weak evidence narrows reliance |
| 4 | Produce the first tax-committee product | Portfolio memorandum, dashboard, exhibits, editable presentation | Bounded internal decisions only |
| 5 | Execute controlled tax diligence | New synthetic evidence, reconciled facts, refreshed confidence | Internal evidence refresh only |
| 6 | Optimize structures under legal and economic constraints | Alternative structures, cash-tax/ETR/NPV scenarios, robustness tests | Design artifact; no implementation |
| 7 | Select counsel, advisers, and valuation specialists | Conflict-aware, independence-aware shortlists | Process design only |
| 8 | Simulate controlled instruction and implementation planning | Recipient verification, disclosure tiers, dry-run implementation log | No transmission or restructuring |
| 9 | Introduce quarterly tax-code intelligence and universe growth | 10 changed codes, 5 new companies, impact map, targeted recommendation reviews | Human review; no automatic V2 |
| 10 | Integrate the read-only operating application | Manifest, permissions, health checks, playbooks, quarterly scheduler | Prototype complete; no production action |

The quarterly loop at Step 9 changes exactly 10% of the 100 tax-code modules, adds five synthetic companies, re-evaluates only affected subgraphs, and preserves every historical recommendation version.
"""
    write_text(vault / "00_System" / "PROJECT_ROADMAP.md", roadmap)

    ontology = {
        "objects": [
            "Jurisdiction", "TaxCode", "Conglomerate", "Entity", "OwnershipEdge",
            "IntercompanyTransaction", "Source", "AtomicClaim", "Contradiction",
            "Recommendation", "Decision", "QuarterlyChange", "AuditRecord",
        ],
        "core_lineage": [
            "TaxCode -> AtomicClaim -> Recommendation -> Decision",
            "Conglomerate -> Entity -> Transaction -> TaxExposure",
            "QuarterlyChange -> AffectedObject -> Review -> RecommendationVersion",
        ],
        "permission_rule": "Evidence may narrow reliance; only a human decision may widen authority.",
    }
    write_text(vault / "00_System" / "ONTOLOGY.json", json.dumps(ontology, indent=2))

    decision = f"""---
object_type: decision
decision_id: DEC-000
date: {TODAY}
status: ACCEPTED_FOR_SYNTHETIC_ARCHITECTURE_TESTING
synthetic: true
---

# DEC-000 - Accept Step 0 Baseline

## Accepted

- A dual-universe vault comprising 100 synthetic tax-code modules across 20 fictional jurisdictions.
- Ten synthetic complex conglomerates with entity, ownership, and intercompany-transaction graphs.
- Stable identifiers, portable Markdown, structured CSV data, deterministic validation, and a versioned baseline quarter.

## Permitted

- Read, browse, organize, validate, and inspect the synthetic vault.
- Design the Step 1 analytical experiment.

## Prohibited

- No tax recommendation, real-jurisdiction inference, filing position, transaction, restructuring, communication, or implementation.
- No use of synthetic content as legal, tax, accounting, valuation, or investment advice.

## Human gate

Only an explicit later decision may accept Recommendation V1 or widen the permitted action set.
"""
    write_text(vault / "10_Decisions" / "DEC-000.md", decision)

    write_text(vault / "14_Hot_Cache" / "CURRENT_STATE.md", f"""
# Current State - 2026-Q3 Baseline

- Active step: 0 of 10
- Tax-code modules: 100
- Fictional jurisdictions: 20
- Conglomerates: 10
- Recommendations: 0
- Current recommendation version: NONE
- Latest decision: [[DEC-000]]
- Next permitted experiment: Step 1, first structure-design loop
- Explicit prohibition: no real tax advice, external action, filing, communication, or restructuring
""")

    write_text(vault / "00_System" / "README.md", f"""
# {PROJECT_NAME}

An Obsidian-compatible, synthetic research prototype for governed multi-jurisdictional tax-structure design.

## Baseline

- 20 fictional jurisdictions
- 100 synthetic tax-code modules (five per jurisdiction)
- 10 synthetic complex conglomerates
- {len(entity_rows)} legal entities
- {len(ownership_rows)} ownership edges
- {len(transaction_rows)} intercompany transactions
- 0 tax-structure recommendations at Step 0

## Foundational separation

The **external universe** contains tax rules and jurisdictions. The **internal universe** contains conglomerates, entities, ownership, functions, risks, assets, and transactions. Links connect the universes without conflating a rule with a company fact.

## Disclaimer

Every name, rule, rate, threshold, entity, transaction, and relationship is fictional. This vault is an architecture-testing artifact and is not tax, legal, accounting, valuation, or investment advice.

See [[PROJECT_ROADMAP]] and [[DEC-000]].
""")

    expected = {
        "jurisdictions": 20, "tax_codes": 100, "conglomerates": 10,
        "entities": len(entity_rows), "ownership_edges": len(ownership_rows),
        "transactions": len(transaction_rows), "recommendations": 0,
    }
    checks = {
        "tax_code_count": len(tax_code_rows) == expected["tax_codes"],
        "jurisdiction_count": len(jurisdiction_rows) == expected["jurisdictions"],
        "conglomerate_count": len(group_rows) == expected["conglomerates"],
        "unique_tax_code_ids": len({r["tax_code_id"] for r in tax_code_rows}) == 100,
        "unique_entity_ids": len({r["entity_id"] for r in entity_rows}) == len(entity_rows),
        "all_ownership_targets_exist": all(
            e in {r["entity_id"] for r in entity_rows}
            for row in ownership_rows for e in (row["parent_entity_id"], row["child_entity_id"])
        ),
        "no_recommendation_files": not any((vault / "09_Recommendations").glob("*.md")),
        "decision_000_exists": (vault / "10_Decisions" / "DEC-000.md").exists(),
    }
    if not all(checks.values()):
        raise AssertionError(checks)

    hashes = {}
    for path in sorted(vault.rglob("*")):
        if path.is_file():
            hashes[str(path.relative_to(vault))] = hashlib.sha256(path.read_bytes()).hexdigest()
    validation = {
        "step": 0, "generated_on": TODAY, "seed": SEED,
        "status": "PASS", "expected": expected, "checks": checks,
        "file_count_before_validation_record": len(hashes), "sha256": hashes,
    }
    write_text(vault / "13_Audit" / "STEP_0_VALIDATION.json", json.dumps(validation, indent=2))
    write_text(vault / "13_Audit" / "STEP_0_SUCCESS.md", "# STEP 0 SUCCESS\n\nDeterministic validation passed. The vault contains no recommendation records.")
    return validation


def build_notebook(path: Path) -> None:
    source = Path(__file__).read_text(encoding="utf-8")
    # Keep the notebook self-contained by embedding the generator after imports.
    code = source.split("if __name__ == \"__main__\":")[0]
    drive_code = """from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
OUTPUT_ROOT = Path('/content/drive/MyDrive/Tax_Planning_ExoBrain_Project/Step_0')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
VAULT_PATH = OUTPUT_ROOT / 'Tax_Planning_ExoBrain_Vault'
print(f'Output folder: {OUTPUT_ROOT}')
"""
    run_code = """validation = build_vault(VAULT_PATH)
print(json.dumps({
    'status': validation['status'],
    'expected': validation['expected'],
    'vault_path': str(VAULT_PATH),
}, indent=2))
"""
    verify_code = """assert validation['status'] == 'PASS'
assert validation['expected']['tax_codes'] == 100
assert validation['expected']['conglomerates'] == 10
assert validation['expected']['recommendations'] == 0
print('STEP 0 SUCCESS: dual-universe synthetic vault built and validated.')
"""
    notebook = {
        "cells": [
            {"cell_type": "markdown", "metadata": {}, "source": [
                "# Tax Planning Exo-Brain - Baby Step 0\n",
                "## Build the dual-universe synthetic vault\n\n",
                "This notebook adapts the Civil Litigation Exo-Brain benchmark to multi-jurisdictional tax planning. It creates 100 synthetic tax-code modules and 10 complex synthetic conglomerates. **No recommendation is produced at Step 0.**\n\n",
                "All jurisdictions, rules, rates, entities, and transactions are fictional. This is not tax or legal advice."
            ]},
            {"cell_type": "markdown", "metadata": {}, "source": [
                "## Design principle\n\n",
                "The external rule universe and internal company universe are stored separately, linked by stable identifiers, and governed by DEC-000. Evidence may later narrow reliance; only a human decision may widen authority."
            ]},
            {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": drive_code.splitlines(True)},
            {"cell_type": "code", "execution_count": None, "metadata": {"collapsed": True}, "outputs": [], "source": code.splitlines(True)},
            {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": run_code.splitlines(True)},
            {"cell_type": "code", "execution_count": None, "metadata": {}, "outputs": [], "source": verify_code.splitlines(True)},
            {"cell_type": "markdown", "metadata": {}, "source": [
                "## Result\n\n",
                "The Google Drive folder now contains an Obsidian-compatible vault, structured CSV files, DEC-000, a hot cache, the 0-10 roadmap, cryptographic hashes, and a deterministic validation report. The next authorized experiment is Baby Step 1."
            ]},
        ],
        "metadata": {
            "colab": {"name": "Baby_Step_0_Tax_Planning_ExoBrain.ipynb", "provenance": []},
            "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
            "language_info": {"name": "python", "version": "3.x"},
        },
        "nbformat": 4, "nbformat_minor": 5,
    }
    path.write_text(json.dumps(notebook, indent=2), encoding="utf-8")




In [ ]:
validation = build_vault(VAULT_PATH)
print(json.dumps({
    'status': validation['status'],
    'expected': validation['expected'],
    'vault_path': str(VAULT_PATH),
}, indent=2))


In [ ]:
assert validation['status'] == 'PASS'
assert validation['expected']['tax_codes'] == 100
assert validation['expected']['conglomerates'] == 10
assert validation['expected']['recommendations'] == 0
print('STEP 0 SUCCESS: dual-universe synthetic vault built and validated.')


## Result

The Google Drive folder now contains an Obsidian-compatible vault, structured CSV files, DEC-000, a hot cache, the 0-10 roadmap, cryptographic hashes, and a deterministic validation report. The next authorized experiment is Baby Step 1.